# Cost-efficiency table

One tidy row per evaluated system: quality metric + cost, for the Pareto chart.

Cost unit is **USD per decision at k=1** — total run cost divided by the number of
instances, then by the judge's sampling multiplier `k`. Dividing out `k` stops the
3-sample judges from being charged for repetitions the baselines never had the option
to make; keeping the *decision* (not the LLM call) as the denominator stops multi-stage
pipelines from being discounted by their cheap scaffolding calls.

Cost is computed over unfiltered instances, metrics over blocklist-filtered ones. Both
the baselines and the sweeps are counted the same way, so the comparison stays fair.

The retrieval evaluator's cost includes collecting its candidates. That happens once
per track, before any judging, so it is a flat term on top of the judge cost rather
than something `k` divides.

In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "output").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.analysis.artifacts import (
    _extract_instances_path,
    _extract_model_from_artifact_dir,
    _extract_raw_outcomes_for_dir,
)
from novelty_eval.analysis.filtering import (
    _load_paper_blocklist,
    _derive_exclude_indices,
    _recompute_pairwise_metrics,
    _recompute_pointwise_metrics,
)
from novelty_eval.ablation.merge_ablation_runs import (
    _derive_pointwise_partner_excluded,
)
from novelty_eval.analysis.stats_one_pass import (
    _POINTWISE_METRIC_KEYS,
    _all_pointwise_metrics,
    _run_bootstrap_multi,
)

BLOCKED_TITLES = _load_paper_blocklist()

In [ ]:
MERGES = [
    "merge_hvh_all_metrics",           # merge_config_hvh.yaml
    "merge_vanilla_all_metrics",       # merge_config_vanilla.yaml
    "merge_test_ai_researcher_hvh",    # merge_test_hvh.yaml
    "merge_ai_researcher_vanilla_ai",  # merge_test_vanilla_ai.yaml
]

# Judge models to keep. Anything else is skipped.
MODELS = {"gpt-5.4", "claude-opus-4-6"}

# config name -> (family, k), where k is the number of judge samples per decision.
# Anything not listed here is skipped.
CONFIGS = {
    "current":             ("ours", 3),
    "mec_k_1":             ("ours", 1),
    "low_judge_reasoning": ("ours", 3),
    "retrieval":           ("ours", 3),
    "ai_researcher_base":  ("si_et_al", 1),
}

# Metric columns carried into the table, per task. The first is the headline one,
# repeated as `value` so the summaries below stay metric-agnostic; the rest are here
# so a chart can be redrawn over any of them without recomputing anything.
METRIC_COLUMNS = {
    "pointwise": ["f1_macro", "f1_pos", "f1_neg", "accuracy"],
    "pairwise":  ["pairwise_accuracy", "pairwise_accuracy_strict", "pairwise_accuracy_no_ties"],
}

# Runs derived by sub-sampling a parent sweep (derive_subrun_artifacts.py). They
# reuse the parent's calls, so they carry no cost report of their own:
# config -> (parent config, cost divisor).
DERIVED = {"mec_k_1": ("current", 3)}

# Pointwise-only baselines. They live outside the ablation sweeps, so they are listed
# by hand. (family, testset, model, artifact_dir); all are single-pass, so k=1.
BASELINES = [
    ("scideator",    "hvh",     "gpt-5.4",          "output/accuracy_test_artifacts/2026-08-20_12-42-17"),
    ("scideator",    "hvh",     "claude-opus-4-6",  "output/accuracy_test_artifacts/2026-08-20_17-55-09"),
    ("scideator",    "vanilla", "gpt-5.4",          "output/accuracy_test_artifacts/2026-08-21_10-11-44"),
    ("scideator",    "vanilla", "claude-opus-4-6",  "output/accuracy_test_artifacts/2026-08-21_15-14-20"),
    ("ai_scientist", "hvh",     "gpt-5.4",          "output/accuracy_test_artifacts/2026-08-22_10-22-25"),
    ("ai_scientist", "hvh",     "claude-opus-4-6",  "output/accuracy_test_artifacts/2026-08-22_12-54-19"),
    ("ai_scientist", "vanilla", "gpt-5.4",          "output/accuracy_test_artifacts/2026-08-22_15-21-34"),
    ("ai_scientist", "vanilla", "claude-opus-4-6",  "output/accuracy_test_artifacts/2026-08-22_16-27-02"),
]

NAME_RE = re.compile(r"^(pairwise|pointwise)-(hvh|vanilla-ai)_(.+)_\d{4}-\d{2}-\d{2}_[\d-]+$")

In [ ]:
def active_artifact_dirs():
    """Artifact dirs the merge configs selected, as (task, testset, config, model, dir)."""
    seen, out = set(), []
    for merge in MERGES:
        summary = ROOT / "output/ablation_sweeps" / merge / "merged_ablation_summary.md"
        paths = re.findall(r"`(output/ablation_sweeps/\S*?/artifacts/[^`]+)`", summary.read_text())
        for rel in paths:
            d = ROOT / rel
            # The summary records where each run sat when it was merged, and that
            # folder may since have been removed (the `_recheck` merges were); the
            # merge keeps its own copy of every run it selected, so fall back to it.
            if not d.is_dir():
                d = summary.parent / "artifacts" / d.name
            if not d.is_dir():
                # Without the dir the model reads as "Unknown" and the run would
                # vanish from the table without a word, so say so here.
                print(f"MISSING artifact dir ({merge}): {rel}")
                continue
            m = NAME_RE.match(d.name)
            if not m or d.name in seen:
                continue
            task, track, rest = m.groups()
            # Model names contain hyphens, so strip the known model off the tail
            # rather than trying to split config and model apart by pattern.
            model = _extract_model_from_artifact_dir(d)
            if model not in MODELS or not rest.endswith(f"-{model}"):
                continue
            config = rest[: -len(model) - 1]
            if config not in CONFIGS:
                continue
            seen.add(d.name)
            out.append((task, "vanilla" if track == "vanilla-ai" else track, config, model, d))
    return out


# Bootstrap resamples for the error bars. The interval moved by less than 0.001
# between 2000 and 10000; at 10000 the whole table takes about five seconds.
N_RESAMPLES = 10000


def pointwise_cis(d, excluded, metrics):
    """95% bootstrap CIs over instances: {ci_low_<metric>, ci_high_<metric>, ...}.
    """
    # Third argument only labels warnings, and no logger is passed.
    outcomes = _extract_raw_outcomes_for_dir(d, "pointwise", "", None)
    if not outcomes:
        return {}
    pred, target = outcomes["_pointwise_pred"], outcomes["_pointwise_target"]
    pids = sorted(p for p in pred if p not in excluded)  # sorted: pairs rows up across systems
    data = (np.array([pred[p] for p in pids]), np.array([target[p] for p in pids]))

    point = dict(zip(_POINTWISE_METRIC_KEYS, _all_pointwise_metrics(*data)))
    for metric in METRIC_COLUMNS["pointwise"]:
        assert abs(point[metric] - metrics[metric]) < 5e-4, (d.name, metric, point[metric], metrics[metric])

    boot = _run_bootstrap_multi(data, _all_pointwise_metrics, _POINTWISE_METRIC_KEYS,
                                n_resamples=N_RESAMPLES, random_state=42,
                                label=d.name, _logger=None)

    return {f"ci_{bound}_{metric}": r[f"ci_{bound}"]
            for metric, r in boot.items() for bound in ("low", "high")}


def quality(d, task):
    """Blocklist-filtered metrics for one run, or None if it has no scores."""
    instances = _extract_instances_path(d)
    excluded = set(_derive_exclude_indices(instances, BLOCKED_TITLES))
    if task == "pointwise":
        excluded |= set(_derive_pointwise_partner_excluded(instances, BLOCKED_TITLES))
        m = _recompute_pointwise_metrics(d, excluded)
        return m if m is None else {**m, **pointwise_cis(d, excluded, m)}
    return _recompute_pairwise_metrics(d, excluded)


def read_cost(d):
    """(cost_usd, instances) for a run, or None if it has no billed calls."""
    f = d / "cost_report.json"
    if not f.exists():
        return None
    c = json.loads(f.read_text())
    n = c.get("instances_processed") or c.get("instances_total")
    return (c["total_cost_usd"], n) if c["total_calls"] and n else None


# Retrieving the candidates happens once per track, before any judging: the judge's
# cost report holds judge calls only, and re-sampling the judge k times re-reads the
# same collected set. So it rides on the retrieval evaluator's cost as a flat term,
# on top of the judge cost that k divides. Both tasks read the one collected set (a
# pointwise instance is one side of a pairwise one), so each is charged for it.
RETRIEVAL_CONFIGS = {"retrieval"}
RETRIEVAL_COLLECTION_DIRS = {
    "hvh": "output/iclr_test_instances/pairwise_data/20260429_221238/manipulated",
    "vanilla": "output/iclr_test_instances/pairwise_data/20260517_171954/manipulated",
}
_RETRIEVAL_COST_RE = re.compile(r"Estimated retrieval cost: \$([\d.]+) \(\d+ LLM calls")
_retrieval_cost_cache = {}


def retrieval_collection_cost(config, testset):
    """USD for one cold pass of retrieve_candidates.py over a track; 0 for other configs.

    Every invocation logs its own total, and these caches were rebuilt repeatedly
    while the pipeline was in flux; a re-run over a warm cache logs near zero, so the
    largest run is the one that collected the track from cold -- what collecting
    these candidates once actually costs.
    """
    if config not in RETRIEVAL_CONFIGS:
        return 0.0
    if testset not in _retrieval_cost_cache:
        d = ROOT / RETRIEVAL_COLLECTION_DIRS[testset]
        costs = [
            float(m.group(1))
            for f in d.glob("logs/**/*")
            if f.is_file()
            for m in _RETRIEVAL_COST_RE.finditer(f.read_text(errors="ignore"))
        ]
        _retrieval_cost_cache[testset] = max(costs, default=0.0)
    return _retrieval_cost_cache[testset]


def row(family, task, testset, config, model, k, d, cost):
    """One tidy row, or None for a run with no scores or no cost."""
    m = quality(d, task)
    if m is None or cost is None:
        return None
    judge_cost, n = cost
    collection = retrieval_collection_cost(config, testset)
    cost_usd = judge_cost + collection
    cost_usd_k1 = judge_cost / k + collection
    columns = METRIC_COLUMNS[task]
    return {
        "family": family,
        "task": task,
        "testset": testset,
        "config": config,
        "model": model,
        "k": k,
        "metric": columns[0],
        "value": m[columns[0]],
        **{c: m[c] for c in columns},
        # Pairwise rows have no CIs, so these stay empty there.
        **{f"ci_{bound}_{c}": m.get(f"ci_{bound}_{c}")
           for c in columns for bound in ("low", "high")},
        "support": m["support"],
        "n": n,
        "cost_usd": cost_usd,
        "cost_usd_k1": cost_usd_k1,
        "cost_per_decision_k1": cost_usd_k1 / n,
        "run_dir": str(d.relative_to(ROOT)),
    }

In [ ]:
found = list(active_artifact_dirs())
measured = {(task, testset, config, model): read_cost(d)
            for task, testset, config, model, d in found}


def cost_of(task, testset, config, model, d):
    """A derived run has no cost report; charge it a share of its parent's total."""
    if config in DERIVED:
        parent, divisor = DERIVED[config]
        base = measured.get((task, testset, parent, model))
        return None if base is None else (base[0] / divisor, base[1])
    return measured[(task, testset, config, model)]


specs = [
    (CONFIGS[config][0], task, testset, config, model, CONFIGS[config][1], d,
     cost_of(task, testset, config, model, d))
    for task, testset, config, model, d in found
] + [
    (family, "pointwise", testset, family, model, 1, ROOT / path, read_cost(ROOT / path))
    for family, testset, model, path in BASELINES
]

rows, dropped = [], []
for spec in specs:
    r = row(*spec)
    (rows.append(r) if r else dropped.append(spec))

for family, task, testset, config, model, *_rest in dropped:
    print(f"DROPPED (no scores or no cost): {task}/{testset}/{config}/{model}")

df = pd.DataFrame(rows).sort_values(
    ["task", "testset", "family", "config", "model"], ignore_index=True
)
df.to_csv(ROOT / "output/cost_efficiency.csv", index=False)
print(f"\n{len(df)} rows -> output/cost_efficiency.csv")
df

In [ ]:
# Coverage check: which (task, testset, config) cells are missing models.
cov = df.pivot_table(index=["task", "testset", "config"], columns="model", values="value", aggfunc="first")
cov.notna().astype(int)

In [ ]:
cols = ["config", "model", "value", "cost_usd_k1", "cost_per_decision_k1", "n", "k"]
for (task, testset), g in df.groupby(["task", "testset"], sort=False):
    print(f"=== {task} / {testset} ({g['metric'].iloc[0]}) ===")
    print(g.sort_values("cost_per_decision_k1")[cols].to_string(index=False, float_format="%.4f"))
    print()

## Cost-quality frontier

One figure per metric and judge model, each on shared axes so its two panels can be
read side by side. Log-x because cost spans ~40x. Colour is the evaluator and shape is
its family: prompted judges are circles, and each external system has a shape of its
own. The figures are pointwise; the pairwise numbers live in the appendix table below.
Edit the `TEXT` dict at the top of the next cell to change any label, or to drop a row
from the figures (the table keeps it either way).

The figures are laid out short and wide: nothing but the panel titles (with the judge
model at their right) and the one-row legend sits outside the panels, and the cost axis
goes unlabelled, since its `$` ticks already name it — say "total cost (log scale)" in
the LaTeX caption instead, which the next cell prints as a reminder.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, NullLocator

# Which cost column goes on the x-axis. "cost_usd" is what each run actually cost.
# The k=1-normalised columns are still in the table, but mec_k_1 is charged a third
# of its parent's total, so under "cost_usd_k1" it lands exactly on top of
# prompted-baseline -- with mec_k_1 plotted, the raw total is the honest axis.
X_COLUMN = "cost_usd"
# Ticks are drawn at every one of these that falls inside a panel's range. Two that
# sit close together on a log axis would collide once labelled, so thin_x_labels()
# below drops the label and keeps the tick.
X_TICKS = [0.5, 1, 2, 5, 10, 20, 40, 80]

# Where the figures land: a subdirectory of output/figures/ named after this one,
# as every other figure notebook does. It is a directory rather than a stem because
# this notebook writes six figures.
FIG_DIR = ROOT / "output/figures/cost-efficiency"

# Short and wide: quality spans about 0.3 while cost spans ~40x, so the panels are
# given their room across rather than up. Everything that is not data sits in two
# bands -- panel titles above, legend below -- and nothing else reserves height.
FIG_SIZE = (7.2, 2.2)
# White margin left around the saved crop. The default 0.1 leaves the legend all but
# touching the edge of the page it lands on.
SAVE_PAD_IN = 0.15

# A short coloured tick on the axis floor under every mark, pinning its cost. The
# tick labels above mostly cover this now; a full drop-line to the floor is not an
# option -- it would read as a bar, off a baseline that means nothing here.
SHOW_COST_RUG = False

# The staircase marking the best score reachable at each budget. With few points it
# can read as a stray elbow, and it draws a hard ordering through gaps that are
# inside the confidence interval -- set False to show the points alone.
SHOW_FRONTIER = True

# ---- everything the chart says, in one place ----------------------------------
# The two dicts below are both the labels and the selection: an evaluator or judge
# model that is not listed here is left out of the figures entirely (the table keeps
# it). Order is legend order.
TEXT = {
    # No longer drawn: the panels name the cost axis through their $ ticks, so this
    # phrase belongs in the LaTeX caption rather than in a label band of its own.
    # It is printed once below, to be pasted there.
    "x_label": "total cost (log scale)",
    "metrics": {                             # per task: table column -> y-axis label
        "pointwise": {"f1_macro": "F1 (macro)",
                      "f1_pos":   "F1 (positive)",
                      "f1_neg":   "F1 (negative)"},
    },
    # Pointwise only: the pairwise figures are gone, and the appendix table below
    # is where the pairwise numbers live now. Only the keys are read -- with one
    # task left there is nothing to tell apart, so the names go unused.
    "tasks": {"pointwise": "pointwise"},
    "panels": {"hvh": "Human-Only", "vanilla": "Human+Generated"},
    "evaluators": {                          # order here is the legend order
        "current":             "Prompted (reference)",
        "mec_k_1":             "Prompted (no aggregation)",
        "low_judge_reasoning": "Prompted (low reasoning)",
        "retrieval":           "Prompted (retrieval)",
        "scideator":           "Idea Novelty Checker",
        "ai_scientist":        "AI-Scientist",
        # Pairwise-only, so it never reaches a figure; the appendix table still
        # reads its name from here.
        "ai_researcher_base":  "AI-Researcher",
    },
    "models": {"claude-opus-4-6": "Opus 4.6", "gpt-5.4": "GPT-5.4"},
}

# Hues are the "Oil" swatches (#404473 indigo, #806292 purple, #bf959b rose,
# #f0cda1 sand) snapped to the nearest step that clears the data-viz checks on a
# white surface: as given they sit under the chroma floor and outside the lightness
# band, so drawn raw they would read as four grays. Checked with --pairs all -- a
# scatter compares every pair of colours, not just adjacent ones. The figures are
# pointwise, so the six marks they hold are what was checked: worst CVD dE 10.1 /
# normal dE 16.1, all at least 3:1 on white.
COLOR = {
    "current":             "#474a8e",  # indigo
    "mec_k_1":             "#757bc4",  # the same prompt at k=1, so the same hue lighter
    "low_judge_reasoning": "#8f204f",  # wine
    "retrieval":           "#c87181",  # rose
    "scideator":           "#845502",  # bronze
    "ai_scientist":        "#aa8c0e",  # ochre
}

# Shape carries the judge family: every prompted judge is a circle, and each
# external system gets a shape of its own. The judge model is no longer a shape --
# each figure holds one model and names it above its right-hand panel.
MARKER = {
    "current":             "o",
    "mec_k_1":             "o",
    "low_judge_reasoning": "o",
    "retrieval":           "o",
    "scideator":           "s",
    "ai_scientist":        "^",
}

# Ink, secondary ink and grid lines are the palette's darkest swatch and steps of
# it. The surface is plain white, so the figure drops onto a white page -- and onto
# a dark viewer -- carrying no tint of its own.
INK, MUTED, GRID, SURFACE = "#23233c", "#5c5a70", "#e7e0d6", "#ffffff"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,  # embed TrueType outlines; most venues reject Type 3
})


def pareto_front(rows, metric):
    """Non-dominated (min cost, max `metric`) rows, cheapest first."""
    best, front = -float("inf"), []
    for r in sorted(rows, key=lambda r: getattr(r, X_COLUMN)):
        if getattr(r, metric) > best:
            front.append(r)
            best = getattr(r, metric)
    return front


def thin_x_labels(fig, ax, gap_pt=5):
    """Hide cost labels that would run into the one before them; ticks still draw."""
    fig.canvas.draw()  # label boxes are only known once the figure has been laid out
    right = -float("inf")
    for label in ax.xaxis.get_ticklabels():
        box = label.get_window_extent()
        if box.x0 < right + gap_pt:
            label.set_visible(False)
        else:
            right = box.x1


def plot_pareto(df, path, task, metric, model, judge_name):
    """One figure per (task, metric, judge model); `TEXT` says everything else."""
    sub = df[(df["task"] == task)
             & df["config"].isin(TEXT["evaluators"])
             & df["model"].isin(TEXT["models"])]
    # Panels and axis limits come from every model of this task, not just the drawn
    # one, so its two figures share a frame and can be read side by side. Each task
    # keeps a frame of its own -- different metric, different cost scale.
    sets = [s for s in TEXT["panels"] if s in set(sub["testset"])]
    xlim = (sub[X_COLUMN].min() / 1.7, sub[X_COLUMN].max() * 1.7)
    ylim = (sub[f"ci_low_{metric}"].min() - 0.02, sub[f"ci_high_{metric}"].max() + 0.02)
    shown = sub[sub["model"] == model]

    fig, axes = plt.subplots(1, len(sets), figsize=FIG_SIZE, sharex=True, sharey=True,
                             facecolor=SURFACE)
    for ax, testset in zip(axes, sets):
        rows = list(shown[shown["testset"] == testset].itertuples())
        ax.set_facecolor(SURFACE)
        ax.set_xscale("log")
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        if SHOW_FRONTIER and rows:
            # Best score reachable at each budget; stops at the last frontier point
            # rather than trailing off to the plot edge.
            front = pareto_front(rows, metric)
            ax.step([getattr(r, X_COLUMN) for r in front], [getattr(r, metric) for r in front],
                    where="post", color=MUTED, lw=1, alpha=0.4, zorder=1)
        rug_top = ylim[0] + 0.03 * (ylim[1] - ylim[0])
        for r in rows:
            x = getattr(r, X_COLUMN)
            if SHOW_COST_RUG:
                ax.vlines(x, ylim[0], rug_top, color=COLOR[r.config], lw=1.4, zorder=2)
            # The interval is asymmetric, so each arm is given its own length.
            y, lo, hi = (getattr(r, metric),
                         getattr(r, f"ci_low_{metric}"), getattr(r, f"ci_high_{metric}"))
            ax.errorbar(x, y, yerr=[[y - lo], [hi - y]], fmt="none", ecolor=COLOR[r.config],
                        elinewidth=1.2, capsize=2.5, alpha=0.7, zorder=2)
            # A surface-coloured ring keeps two marks apart where they overlap.
            ax.plot(x, y, MARKER[r.config], ms=8,
                    mfc=COLOR[r.config], mec=SURFACE, mew=1.6, zorder=3)

        ax.set_xticks([t for t in X_TICKS if xlim[0] <= t <= xlim[1]])
        ax.xaxis.set_minor_locator(NullLocator())
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:g}"))
        ax.set_xlim(*xlim)  # again: set_xticks widens the view to fit its ticks
        # No x-axis label: it would be drawn once per panel for an axis the $ ticks
        # already name, and its band is the tallest thing under the panels.
        ax.set_title(TEXT["panels"][testset], fontsize=9, color=INK, loc="left", pad=3)
        ax.grid(True, which="major", color=GRID, lw=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(which="both", labelsize=7.5, colors=MUTED, length=0, pad=2)
        # Stubs on the cost axis only: they read as a ruler under the marks, where
        # on the quality axis the horizontal grid lines already do that job.
        ax.tick_params(axis="x", which="major", length=3.5, color=GRID)
        for side, s in ax.spines.items():
            s.set_visible(side in ("left", "bottom"))
            s.set_color(GRID)

    axes[0].set_ylabel(TEXT["metrics"][task][metric], fontsize=8, color=MUTED, labelpad=2)
    # The judge model rides the far right of the panel-title band, in the secondary
    # ink: it names the whole figure, which is why it is not a suptitle -- a title
    # of its own would cost a band for one short phrase. The metric is on the y-axis
    # and the task is the only one plotted, so neither needs saying again.
    axes[-1].set_title(judge_name, fontsize=9, color=MUTED, loc="right", pad=3)

    seen = set(shown["config"])
    handles = [Line2D([], [], marker=MARKER[key], ls="", ms=8, mfc=COLOR[key],
                      mec=SURFACE, mew=1.6, label=label)
               for key, label in TEXT["evaluators"].items() if key in seen]
    # One row, so the legend costs a single line of height. At these six labels it
    # measures 7.0in against the figure's 7.2in; a longer name would push past the
    # panels and widen the saved crop, so spend any new label on `fontsize` or
    # `columnspacing` before adding it. The offset is what keeps the row clear of
    # the cost labels above it -- closer than about -0.04 and the two bands merge.
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False,
               fontsize=7, labelcolor=MUTED, bbox_to_anchor=(0.5, -0.06),
               handletextpad=0.3, columnspacing=1.0)

    # No rect: the legend hangs outside the figure and the saved crop is tight, so
    # bands reserved in here would only become dead margin inside that crop.
    fig.tight_layout()
    for ax in axes:
        thin_x_labels(fig, ax)
    fig.savefig(path, bbox_inches="tight", pad_inches=SAVE_PAD_IN, dpi=200, facecolor=SURFACE)
    fig.savefig(str(path).replace(".png", ".pdf"), bbox_inches="tight",
                pad_inches=SAVE_PAD_IN, facecolor=SURFACE)
    # No return: the inline backend draws the open figures once at cell end.


FIG_DIR.mkdir(parents=True, exist_ok=True)
for task in TEXT["tasks"]:
    for metric in TEXT["metrics"][task]:
        for judge, judge_name in TEXT["models"].items():
            plot_pareto(df, FIG_DIR / f"pareto_{task}_{metric}_{judge}.png", task, metric,
                        judge, judge_name)

print(f"wrote {2 * sum(len(m) for m in TEXT['metrics'].values()) * len(TEXT['models'])} files "
      f"to {FIG_DIR.relative_to(ROOT)}/")
print(f"for the figure captions, the x axis is: {TEXT['x_label']}")


## Appendix tables

The numbers behind the figures, as LaTeX: one table per task, rows grouped by test
set and sorted by evaluator, then judge. Best value per quality column is bolded
within each test set; cost columns are left plain, since lower is better there.


In [ ]:
# Appendix tables: the numbers behind the figures above, one table per task.
# Names come from `TEXT`, but membership comes from `df` -- an evaluator dropped
# from the figures still gets its row here.

# task -> (column, header) pairs, in table order.
TABLE_METRICS = {
    "pointwise": [
        ("f1_macro", r"\textbf{F$_1$}"),
        ("f1_pos", r"\textbf{F$_1^{+}$}"),
        ("f1_neg", r"\textbf{F$_1^{-}$}"),
    ],
    "pairwise": [
        ("pairwise_accuracy", r"\textbf{Acc.}"),
        ("pairwise_accuracy_strict", r"\textbf{Acc.$_{\text{strict}}$}"),
        ("pairwise_accuracy_no_ties", r"\textbf{Acc.$_{\text{no ties}}$}"),
    ],
}

COST_NOTE = (
    "Cost is what the run actually billed, over $k$ judge samples per decision. The "
    "1-sample prompted judge reuses its parent run's calls, so it is charged a third "
    "of that run's total, and the retrieval evaluator's cost includes collecting its "
    "candidates, done once per track."
)
CI_NOTE = (
    rf" Subscripts are the half-width of the 95\% bootstrap confidence interval over "
    rf"test instances, from {N_RESAMPLES:,} resamples. The interval is close enough to "
    rf"symmetric to report one number: its two arms differ by at most 0.012."
)
CAPTIONS = {
    "pointwise": "Pointwise quality and cost of every evaluator, on both test sets. "
                 + COST_NOTE + CI_NOTE,
    "pairwise": "Pairwise quality and cost of every evaluator, on both test sets. " + COST_NOTE,
}
LABELS = {"pointwise": "tab:cost_pointwise", "pairwise": "tab:cost_pairwise"}


def metric_cell(row, metric, best) -> str:
    r"""`0.806`, or `0.806_{\pm0.046}` where the row carries a CI."""
    score = getattr(row, metric)
    value = f"{score:.3f}"
    lo, hi = getattr(row, f"ci_low_{metric}"), getattr(row, f"ci_high_{metric}")
    if pd.isna(lo):
        return rf"\textbf{{{value}}}" if score == best else value
    body = rf"\mathbf{{{value}}}" if score == best else value
    return rf"${body}_{{\pm{(hi - lo) / 2:.3f}}}$"


def latex_cost_table(df, task, metrics=None, caption=None, label=None) -> str:
    """LaTeX table of one task: a row per (evaluator, judge), grouped by test set.

    Rows are sorted by evaluator name, then by judge; the best value per quality
    column is bolded within each test set.
    """
    metrics = metrics or TABLE_METRICS[task]
    caption = caption or CAPTIONS[task]
    label = label or LABELS[task]
    sub = df[df["task"] == task]
    n_cols = len(metrics) + 4

    lines = [
        # `table*` spans both columns of a two-column layout, and `tabular*` with a
        # fill between columns stretches the body across the full text width.
        r"\begin{table*}[t]",
        r"\centering",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{4pt}",
        # The CI subscripts hang below the baseline, so the rows need a little leading.
        r"\renewcommand{\arraystretch}{1.15}",
        rf"\begin{{tabular*}}{{\textwidth}}{{@{{\extracolsep{{\fill}}}}"
        rf"llc{'c' * len(metrics)}r}}",
        r"\toprule",
        " & ".join(
            [r"\textbf{Evaluator}", r"\textbf{Judge}", r"\textbf{$k$}"]
            + [h for _, h in metrics]
            + [r"\textbf{Cost (USD)}"]
        )
        + r" \\",
    ]
    for i, (testset, group) in enumerate(sub.groupby("testset", sort=False)):
        lines.append(r"\midrule" if i == 0 else r"\addlinespace")
        name = TEXT["panels"].get(testset, testset)
        lines.append(rf"\multicolumn{{{n_cols}}}{{l}}{{\textit{{{name}}}}} \\")
        best = {metric: group[metric].max() for metric, _ in metrics}
        rows = sorted(
            group.itertuples(),
            key=lambda r: (TEXT["evaluators"].get(r.config, r.config), r.model),
        )
        for row in rows:
            cells = [metric_cell(row, metric, best[metric]) for metric, _ in metrics]
            lines.append(
                " & ".join(
                    [
                        TEXT["evaluators"].get(row.config, row.config),
                        rf"\texttt{{{row.model}}}",
                        str(row.k),
                    ]
                    + cells
                    + [f"{row.cost_usd:.2f}"]
                )
                + r" \\"
            )
    lines += [
        r"\bottomrule",
        r"\end{tabular*}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table*}",
    ]
    return "\n".join(lines)


for task in TABLE_METRICS:
    # A `%` prefix keeps the banner visible here and inert if it is copied along.
    print(f"% {'=' * 20} {task} -> {LABELS[task]} {'=' * 20}")
    print(latex_cost_table(df, task))
    print()
